# Milestone 0.4: Python Optimization - Memory Management

This notebook focuses on **memory management** in Python, particularly relevant when dealing with large datasets common in AI/ML.

**Goal:** Understand how different Python objects consume memory, compare memory usage of lists vs. NumPy arrays, and learn about memory-efficient techniques like generators.

**Reference Script:** `memory_management.py`

## Memory in Python

Python manages memory automatically using reference counting and a garbage collector. However, understanding how much memory different data structures use is crucial for avoiding `MemoryError` exceptions and writing efficient code.

*   **`sys.getsizeof()`:** This built-in function gives the size of a Python object in bytes. **Important:** For container objects like lists, it often gives the size of the container itself, *not* the total size including all the contained elements. Its accuracy varies depending on the object type.
*   **NumPy `.nbytes`:** NumPy arrays have a `.nbytes` attribute that accurately reports the total memory consumed by the array's elements (data buffer).
*   **Garbage Collection (`gc` module):** Python automatically reclaims memory from objects that are no longer referenced. The `gc` module allows interacting with the garbage collector, e.g., manually triggering collection with `gc.collect()`.

In [5]:
import sys
import numpy as np
import gc
import time # For potential timing of memory-intensive operations

# Import functions from the script
try:
    from phase0.python_optimization.memory_management import (
        create_large_list,
        create_large_numpy_array,
        process_data_generator,
        data_generator
    )
except ImportError:
    print("Error: Could not import from memory_management.py. Make sure it's in the correct path.")
    # Define fallbacks if necessary for demonstration
    def create_large_list(size): return list(range(size))
    def create_large_numpy_array(size): return np.arange(size, dtype=np.int64)
    def data_generator(size): 
        for i in range(size): yield i
    def process_data_generator(gen): 
        count = 0
        total = 0
        for item in gen: 
            total += item
            count += 1
        print(f"Processed {count} items from generator.")

## Comparing Memory: Lists vs. NumPy Arrays

Let's create a large list and a NumPy array with the same number of elements and compare their memory footprint.

In [6]:
data_size = 10_000_000 # 10 million elements

print(f"Using data size: {data_size:,}")

# --- Python List ---
print("--- Python List ---")
start_time = time.time()
my_list = create_large_list(data_size) # Function from script prints size
end_time = time.time()
# Manually print size using sys.getsizeof here as well
list_memory_bytes = sys.getsizeof(my_list)
print(f"sys.getsizeof(my_list): {list_memory_bytes:,} bytes (~{list_memory_bytes / (1024*1024):.2f} MB)")
print(f"List creation time: {end_time - start_time:.4f} seconds")

# --- NumPy Array ---
print("NumPy Array ---")
start_time = time.time()
my_array = create_large_numpy_array(data_size) # Function from script prints size
end_time = time.time()
# Manually print size using .nbytes
array_memory_bytes = my_array.nbytes
print(f"my_array.nbytes: {array_memory_bytes:,} bytes (~{array_memory_bytes / (1024*1024):.2f} MB)")
print(f"Array creation time: {end_time - start_time:.4f} seconds")

Using data size: 10,000,000
--- Python List ---

Creating a list with 10,000,000 integers...
Approximate memory usage of the list object: 80,000,056 bytes (~76.29 MB)
sys.getsizeof(my_list): 80,000,056 bytes (~76.29 MB)
List creation time: 0.2142 seconds
NumPy Array ---

Creating a NumPy array with 10,000,000 integers...
Accurate memory usage of the NumPy array data: 80,000,000 bytes (~76.29 MB)
my_array.nbytes: 80,000,000 bytes (~76.29 MB)
Array creation time: 0.0397 seconds


**Observations:**

*   **Memory:** Notice that the NumPy array (`my_array.nbytes`) usually consumes significantly less memory than the Python list (`sys.getsizeof(my_list)`) for the same number of numerical elements. This is because NumPy arrays store elements in a contiguous block of memory with minimal overhead per element, while Python lists store pointers to objects, incurring more overhead.
*   **`sys.getsizeof()` Limitation:** The value from `sys.getsizeof(my_list)` primarily reflects the size of the list's internal buffer of pointers, not the sum of the sizes of all the integer objects it points to. The actual memory consumed by the list *and* its contents is typically larger than what `sys.getsizeof` reports directly for the list object itself. `.nbytes` for NumPy arrays is more accurate for the data buffer.

## Releasing Memory and Garbage Collection

When objects are no longer needed, Python's garbage collector eventually reclaims their memory. We can remove references using `del` and suggest collection using `gc.collect()`.

In [7]:
print("--- Releasing Memory ---")
print(f"Memory before del (approx List): {sys.getsizeof(my_list)/(1024*1024):.2f} MB")
print(f"Memory before del (Array .nbytes): {my_array.nbytes/(1024*1024):.2f} MB")

# Remove references
del my_list
del my_array

print("References deleted.")

# Suggest garbage collection
print("Suggesting garbage collection...")
collected_count = gc.collect() # Returns the number of unreachable objects collected
print(f"Garbage collector finished. Objects collected: {collected_count}")

# Note: Measuring actual memory release requires OS tools (Task Manager, htop, etc.) 
# Python might not immediately return the memory to the OS.

--- Releasing Memory ---
Memory before del (approx List): 76.29 MB
Memory before del (Array .nbytes): 76.29 MB
References deleted.
Suggesting garbage collection...
Garbage collector finished. Objects collected: 0


## Memory-Efficient Processing: Generators

When processing sequences of data, creating large intermediate lists can consume a lot of memory (as seen in `efficient_pipelines.py`). Generators provide a memory-efficient alternative by producing items one at a time, on demand.

Let's use the `data_generator` and `process_data_generator` functions.

In [8]:
large_data_size = 20_000_000 # Process even more data

print(f"--- Processing with Generator (size={large_data_size:,}) ---")

# Create the generator (doesn't compute anything yet)
my_generator = data_generator(large_data_size)
print(f"Generator object created: {my_generator}")
print(f"Memory size of generator object itself: {sys.getsizeof(my_generator)} bytes")
# Note the tiny size of the generator object itself!

# Process the data - computation happens here
start_time = time.time()
process_data_generator(my_generator) # Function from script prints progress/results
end_time = time.time()
print(f"Generator processing time: {end_time - start_time:.4f} seconds")

# Compare this to the memory required if we created a list/array of this size first!
# estimated_list_mb = (sys.getsizeof(list(range(100)))/100 * large_data_size) / (1024*1024) # Very rough estimate
# estimated_array_mb = (np.int64().itemsize * large_data_size) / (1024*1024)
# print(f"Estimated memory if using list: > {estimated_list_mb:.0f} MB")
# print(f"Estimated memory if using array: ~ {estimated_array_mb:.0f} MB")

--- Processing with Generator (size=20,000,000) ---
Generator object created: <generator object data_generator at 0x7f80d68d5630>
Memory size of generator object itself: 208 bytes

Processing data using a generator (memory-efficient)...
Finished processing 20000000 items. Final sum: 399999980000000
Generator processing time: 1.2374 seconds


## Conclusion

*   **NumPy Arrays:** Generally more memory-efficient than Python lists for numerical data.
*   **`sys.getsizeof` vs `.nbytes`:** Understand the limitations of `sys.getsizeof` for containers; use `.nbytes` for NumPy data buffers.
*   **Generators:** Excellent for processing large sequences of data without loading everything into memory at once.
*   **Large Data:** For datasets that don't fit in RAM, explore techniques like chunking (e.g., with Pandas) or memory-mapping (`numpy.memmap`), or distributed computing frameworks (like Dask or Spark).